# MLflow Experiment Tracking y Model Registry

## 🎯 Objetivo

Dominar **MLflow** para el ciclo completo de MLOps:

* 📋 **Tracking**: Experimentos, métricas, hiperparámetros
* 📦 **Model Registry**: Versionado y lifecycle management
* 🚀 **Deployment**: Batch, real-time, streaming
* 📊 **Monitoring**: Drift detection y retraining
* 💾 **Feature Store**: Repositorio centralizado

---

## 🔄 Workflow MLOps Completo

```
📊 EDA/Feature Eng
    ↓
📋 TRACK experimentos
    ↓
⚖️  COMPARE modelos
    ↓
📦 REGISTER mejor modelo
    ↓
🚀 DEPLOY a producción
    ↓
📊 MONITOR performance
    ↓
🔄 RETRAIN cuando sea necesario
```

---

## 🧪 MLflow: Componentes Principales

### 1️⃣ MLflow Tracking

**Registra**:
* Parámetros (hiperparámetros del modelo)
* Métricas (accuracy, RMSE, etc.)
* Modelos (archivos serializados)
* Artefactos (gráficos, datasets)
* Metadata (fecha, usuario, código)

### 2️⃣ MLflow Models

**Formato estándar** para empaquetar modelos:
* Cualquier framework (sklearn, PyTorch, TensorFlow)
* Inferencia consistente
* Dependencias incluidas

### 3️⃣ MLflow Model Registry

**Gestiona ciclo de vida**:
```
None → Staging → Production → Archived
```

### 4️⃣ MLflow Projects

**Empaqueta código ML** reproducible:
* Entry points definidos
* Dependencias declaradas
* Ejecutable en cualquier plataforma

---

## 📚 Lo que Aprenderemos

1. ✅ Tracking de múltiples experimentos
2. ✅ Comparar runs en MLflow UI
3. ✅ Registrar y versionar modelos
4. ✅ Promover modelos a producción
5. ✅ Deployment patterns
6. ✅ Monitoreo y retraining
7. ✅ Feature Store integration

Empecemos con tracking!

## 📋 MLflow Tracking - Ejemplo Práctico

### Scenario: Predecir Churn con 3 Configuraciones

Vamos a entrenar 3 modelos Random Forest con diferentes hiperparámetros y **trackear todo con MLflow**.

### 📦 Lo que MLflow Registrará

* **Parámetros**: `n_estimators`, `max_depth`
* **Métricas**: `accuracy`, `f1_score`
* **Modelo**: Archivo serializado
* **Metadata**: Fecha, duración, código

### 🎯 Objetivo

Al final veremos los 3 runs en la **MLflow UI** y podremos compararlos.

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("┌──────────────────────────────────────────────┐")
print("│        MLFLOW TRACKING - EXPERIMENT DEMO        │")
print("└──────────────────────────────────────────────┘")

# Configurar experiment
mlflow.set_experiment("/Users/cristiandarioortega@gmail.com/ml_tracking_demo")
print("\n📊 Experiment configurado: /Users/cristiandarioortega@gmail.com/ml_tracking_demo")

# Generar dataset sintético
np.random.seed(42)
n = 1000
X = np.random.randn(n, 10)
y = (X[:, 0] + X[:, 1] + np.random.randn(n) * 0.1 > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n📦 Dataset creado: {len(X_train)} train, {len(X_test)} test")
print(f"   Distribución: {(y_train == 1).sum()}/{len(y_train)} positivos ({(y_train == 1).mean():.1%})")

# Entrenar 3 modelos con diferentes configuraciones
configs = [
    {'n_estimators': 50, 'max_depth': 5, 'name': 'Small'},
    {'n_estimators': 100, 'max_depth': 10, 'name': 'Medium'},
    {'n_estimators': 200, 'max_depth': 15, 'name': 'Large'}
]

print("\n🚀 Entrenando 3 modelos y trackeando con MLflow...\n")

results = []
for i, config in enumerate(configs, 1):
    with mlflow.start_run(run_name=f"rf_model_{config['name']}"):
        # Log parameters
        params = {k: v for k, v in config.items() if k != 'name'}
        mlflow.log_params(params)
        
        # Entrenar
        model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        # Evaluar
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        
        # Log metrics
        mlflow.log_metrics({
            'accuracy': accuracy,
            'f1_score': f1,
            'precision': precision,
            'recall': recall
        })
        
        # Log model
        mlflow.sklearn.log_model(model, "random_forest_model")
        
        # Guardar para comparación
        results.append({
            'Model': config['name'],
            'n_estimators': config['n_estimators'],
            'max_depth': config['max_depth'],
            'Accuracy': accuracy,
            'F1 Score': f1,
            'Precision': precision,
            'Recall': recall
        })
        
        print(f"  ✅ Run {i}/{len(configs)}: {config['name']} - F1={f1:.4f}, Accuracy={accuracy:.4f}")

print("\n┌──────────────────────────────────────────────┐")
print("│            COMPARACIÓN DE MODELOS                │")
print("└──────────────────────────────────────────────┘")

df_results = pd.DataFrame(results)
print("\n" + df_results.to_string(index=False))

best_model = df_results.loc[df_results['F1 Score'].idxmax()]
print(f"\n\n🏆 GANADOR: {best_model['Model']}")
print(f"   F1 Score: {best_model['F1 Score']:.4f}")
print(f"   Accuracy: {best_model['Accuracy']:.4f}")

print("\n✅ ¡Todos los runs han sido registrados en MLflow!")
print("\n👁️  Para ver los experimentos:")
print("   1. Menú izquierdo → 'Experiments'")
print("   2. Busca '/Users/cristiandarioortega@gmail.com/ml_tracking_demo'")
print("   3. Compara los 3 runs lado a lado")
print("   4. Visualiza métricas, parámetros, y modelos")

## 📦 MLflow Model Registry

### 🎯 Lifecycle Management

El **Model Registry** gestiona versiones y estados de modelos en producción.

#### Estados (Stages)

```
🏪 WORKFLOW:

1️⃣ None (Staging Area)
   └─ Modelo recién registrado
   
2️⃣ Staging
   ├─ Pruebas internas
   ├─ Validación con stakeholders
   └─ A/B testing en subset
   
3️⃣ Production
   ├─ Sirviendo tráfico real
   ├─ Monitoreo activo
   └─ SLA crítico
   
4️⃣ Archived
   └─ Descartado pero preservado
```

---

### 💻 Operaciones Principales

#### Registrar Modelo

```python
# Durante training
with mlflow.start_run():
    mlflow.sklearn.log_model(model, "model")
    run_id = mlflow.active_run().info.run_id

# Registrar en Model Registry
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri, name="churn_prediction_model")
```

#### Transiciones de Stage

```python
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Promover a Staging
client.transition_model_version_stage(
    name="churn_prediction_model",
    version=2,
    stage="Staging"
)

# Promover a Production (después de validación)
client.transition_model_version_stage(
    name="churn_prediction_model",
    version=2,
    stage="Production",
    archive_existing_versions=True  # Archivar versión anterior
)
```

#### Cargar Modelo por Stage

```python
# Cargar última versión en Production
model = mlflow.pyfunc.load_model(
    model_uri="models:/churn_prediction_model/Production"
)

predictions = model.predict(new_data)
```

---

### ✅ Ventajas del Model Registry

* **Versionado automático**: Cada registro crea nueva versión
* **Trazabilidad**: Desde datos de entrenamiento hasta producción
* **Control de acceso**: Permisos por usuario/rol
* **Annotations**: Comentarios y metadata custom
* **Webhooks**: Automatizar CI/CD en transiciones
* **Rollback fácil**: Volver a versión anterior instantáneamente

---

### 📊 Ejemplo: Historial de Versiones

```
Model: churn_prediction_model

├── v1 (Archived)     ← Baseline inicial (F1=0.72)
├── v2 (Archived)     ← Con feature engineering (F1=0.78)
├── v3 (Production)   ← Con SMOTE (F1=0.85) ⭐
└── v4 (Staging)      ← Con ensemble (F1=0.87) ← En validación
```

**Cada versión tiene**:
* Link al MLflow Run original
* Artifacts (modelo + dependencias)
* Metadata (creador, fecha, descripción)
* Stage history (quién promovió y cuándo)

## 🚀 Model Serving - Opciones de Deployment

### 1️⃣ Batch Inference

**Uso**: Predicciones periódicas sobre grandes volúmenes

```python
import mlflow.pyfunc

# Cargar modelo de producción
model = mlflow.pyfunc.load_model(
    "models:/churn_prediction_model/Production"
)

# Cargar datos desde Unity Catalog
df = spark.table("main.marketing.customers")

# Inferencia en batch
predictions_pandas = model.predict(df.toPandas())

# Guardar resultados
df_predictions = df.withColumn(
    "churn_probability", 
    predictions_pandas
)
df_predictions.write.mode("overwrite").saveAsTable("main.marketing.churn_predictions")
```

**Ventajas**:
* ✅ Eficiente para grandes volúmenes
* ✅ No requiere infraestructura always-on
* ✅ Fácil de schedulear (Databricks Jobs)

**Cuándo usar**:
* Reportes diarios/semanales
* Scoring de toda la base de clientes
* Recomendaciones periódicas

---

### 2️⃣ Real-Time REST API Endpoint

**Uso**: Predicciones instantáneas bajo demanda

#### Crear Endpoint

```python
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

# Deploy modelo como REST API
endpoint = client.create_endpoint(
    name="churn_prediction_api",
    config={
        "served_models": [{
            "model_name": "churn_prediction_model",
            "model_version": "3",
            "workload_size": "Small",
            "scale_to_zero_enabled": True
        }]
    }
)
```

#### Consumir Endpoint

```python
import requests
import os

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints/churn_prediction_api/invocations"

data = {
    "dataframe_records": [
        {"age": 35, "tenure": 24, "monthly_charges": 75.5},
        {"age": 42, "tenure": 6, "monthly_charges": 105.0}
    ]
}

response = requests.post(
    url,
    json=data,
    headers={"Authorization": f"Bearer {token}"}
)

predictions = response.json()
print(predictions)  # [{"churn": 0.15}, {"churn": 0.72}]
```

**Ventajas**:
* ✅ Latencia ultra-baja (<100ms)
* ✅ Autoscaling automático
* ✅ Scale-to-zero (ahorro de costos)
* ✅ A/B testing entre versiones
* ✅ Monitoreo de SLA integrado

**Cuándo usar**:
* Aplicaciones web/mobile
* Decisiones en tiempo real (aprobar préstamo, detección fraude)
* Chatbots / asistentes

---

### 3️⃣ Streaming Inference

**Uso**: Predicciones sobre streams de datos continuos

```python
from pyspark.sql.functions import struct, col

# Cargar modelo como UDF
model_udf = mlflow.pyfunc.spark_udf(
    spark, 
    model_uri="models:/churn_prediction_model/Production",
    result_type="double"
)

# Stream de eventos
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("/mnt/data/customer_events/")
)

# Aplicar modelo en streaming
predictions_stream = stream_df.withColumn(
    "churn_probability",
    model_udf(struct("age", "tenure", "monthly_charges"))
)

# Escribir resultados
predictions_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/mnt/checkpoints/churn_stream") \
    .table("main.streaming.churn_predictions")
```

**Ventajas**:
* ✅ Procesamiento continuo
* ✅ Baja latencia end-to-end
* ✅ Escalable a millones de eventos/seg
* ✅ Exactamente una vez (exactly-once)

**Cuándo usar**:
* IoT y sensores
* Clickstream analytics
* Monitoreo en tiempo real
* Alertas instantáneas

---

### 📊 Comparación

| Característica | Batch | Real-Time API | Streaming |
|-----------------|-------|---------------|------------|
| **Latencia** | Minutos-horas | <100ms | Segundos |
| **Throughput** | Muy alto | Medio | Alto |
| **Costo** | Bajo | Medio-Alto | Medio |
| **Complejidad** | Baja | Media | Alta |
| **Uso típico** | Reportes | Apps web | IoT/Events |

### 💡 Recomendación

**Híbrido**: Batch para scoring masivo + Real-time API para casos individuales.

## 📊 Monitoreo de Modelos en Producción

### ⚠️ El Problema: Model Degradation

Los modelos en producción **degradan con el tiempo** por:

#### 1️⃣ Data Drift

**Distribución de features cambia**

```
TRAINING (2024):
Edad promedio = 35 años

PRODUCTION (2026):
Edad promedio = 42 años  ← ¡Drift!
```

**Impacto**: Modelo asume distribuciones viejas.

#### 2️⃣ Concept Drift

**Relación X → Y cambia**

```
ANTES (2024):
Contratos mensuales → 40% churn

AHORA (2026):
Contratos mensuales → 20% churn  ← Mejor retención
```

**Impacto**: Modelo sobre-predice churn.

#### 3️⃣ Schema Drift

**Features cambian estructura**

```
ANTES: monthly_charges (float)
AHORA: monthly_charges (string) ← ¡Error!
```

---

### 🔍 Detectar Drift

#### Test Kolmogorov-Smirnov (Numérico)

```python
from scipy.stats import ks_2samp
import pandas as pd

# Comparar distribución train vs producción
train_feature = train_df['monthly_charges']
prod_feature = prod_df['monthly_charges']

statistic, p_value = ks_2samp(train_feature, prod_feature)

if p_value < 0.05:
    print("⚠️  DRIFT DETECTADO en monthly_charges")
    print(f"   P-value: {p_value:.4f}")
    print(f"   Media train: {train_feature.mean():.2f}")
    print(f"   Media prod: {prod_feature.mean():.2f}")
    print("   → Considerar reentrenar modelo")
else:
    print("✅ Distribución estable")
```

#### Chi-Squared Test (Categórico)

```python
from scipy.stats import chisquare

train_counts = train_df['contract_type'].value_counts(normalize=True)
prod_counts = prod_df['contract_type'].value_counts(normalize=True)

statistic, p_value = chisquare(
    prod_counts.values,
    train_counts.values
)

if p_value < 0.05:
    print("⚠️  DRIFT DETECTADO en contract_type")
```

#### Monitoreo de Métricas

```python
import mlflow

# Log métricas de producción diariamente
with mlflow.start_run(run_name="prod_monitoring_2026_07_28"):
    # Métricas actuales
    mlflow.log_metrics({
        "prod_accuracy": current_accuracy,
        "prod_f1": current_f1,
        "data_drift_score": drift_score
    })
    
    # Alerta si cae >5%
    if current_f1 < baseline_f1 * 0.95:
        print("⚠️  ALERTA: F1 cayó >5%")
        print("   → Triggering retraining pipeline")
```

---

### 🔄 Pipeline de Retraining Automático

```python
def automated_retraining_pipeline():
    """Pipeline completo de retraining."""
    
    # 1. Detectar drift
    drift_detected = check_drift(
        train_data=historical_df,
        prod_data=recent_prod_df
    )
    
    if not drift_detected:
        print("✅ No drift - Skip retraining")
        return
    
    print("⚠️  Drift detectado - Iniciando retraining")
    
    # 2. Recolectar datos frescos (incluyendo producción)
    fresh_data = spark.sql("""
        SELECT * FROM main.customers
        WHERE created_date > CURRENT_DATE - INTERVAL 90 DAYS
    """)
    
    # 3. Re-entrenar modelo
    with mlflow.start_run(run_name="auto_retrain"):
        new_model = train_model(fresh_data)
        mlflow.sklearn.log_model(new_model, "model")
        
        # Evaluar en holdout set
        f1_new = evaluate_model(new_model, holdout_data)
        mlflow.log_metric("f1_score", f1_new)
        
        # Comparar con producción actual
        current_model = mlflow.pyfunc.load_model(
            "models:/churn_prediction_model/Production"
        )
        f1_current = evaluate_model(current_model, holdout_data)
        
        # 4. A/B test si mejor
        if f1_new > f1_current:
            print(f"✅ Nuevo modelo mejor: {f1_new:.4f} > {f1_current:.4f}")
            
            # Registrar en Model Registry
            model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
            mlflow.register_model(model_uri, "churn_prediction_model")
            
            # Promover a Staging para validación
            client = MlflowClient()
            versions = client.search_model_versions("name='churn_prediction_model'")
            latest_version = max([int(v.version) for v in versions])
            
            client.transition_model_version_stage(
                name="churn_prediction_model",
                version=latest_version,
                stage="Staging"
            )
            
            print("→ Modelo en Staging - Listo para A/B test")
        else:
            print(f"⚠️  Nuevo modelo NO mejor: {f1_new:.4f} <= {f1_current:.4f}")
            print("→ Mantener modelo actual")
    
    # 5. Notificar equipo
    send_slack_notification(
        f"Retraining completado. Nuevo F1: {f1_new:.4f}"
    )

# Schedulear con Databricks Jobs (diario)
automated_retraining_pipeline()
```

---

### 📊 Dashboard de Monitoreo

**Métricas clave a trackear**:

1. **Performance**:
   - Accuracy, F1, Precision, Recall
   - Por segmento (edad, región, etc.)
   
2. **Data Quality**:
   - % Missing values
   - Outliers detectados
   - Schema violations
   
3. **Data Drift**:
   - KS test p-values por feature
   - Distribución shift visual
   
4. **Operational**:
   - Latencia p95/p99
   - Throughput (requests/sec)
   - Error rate
   
5. **Business**:
   - Falsos positivos (costo)
   - Falsos negativos (churn no detectado)
   - ROI del modelo

---

### ✅ Best Practices

1. **Monitorear diariamente**, actuar cuando sea necesario
2. **Guardar predicciones** para audit trail
3. **A/B test** antes de promover a producción
4. **Rollback plan** instantáneo (volver a versión anterior)
5. **Human-in-the-loop** para decisiones críticas

## 💾 Databricks Feature Store

### 🎯 ¿Qué es Feature Store?

**Repositorio centralizado de features** para ML.

#### Problema que resuelve

```
❌ SIN Feature Store:

Data Scientist A:              Data Scientist B:
customer_ltv = ...            customer_lifetime_value = ...
(Cálculo ligeramente diferente)

→ Inconsistencia entre modelos
→ Duplicación de código
→ No reutilización
```

```
✅ CON Feature Store:

Feature Table: customer_features
├─ customer_ltv (definición única)
├─ avg_session_time
└─ purchase_frequency

→ Reutilización
→ Consistencia
→ Trazabilidad
```

---

### 🛠️ Crear Feature Table

```python
from databricks.feature_store import FeatureStoreClient
from pyspark.sql import functions as F

fs = FeatureStoreClient()

# Calcular features
customer_features = spark.sql("""
    SELECT 
        customer_id,
        AVG(session_duration_sec) AS avg_session_time,
        COUNT(DISTINCT order_id) AS purchase_count,
        SUM(order_amount) AS total_spend,
        DATEDIFF(CURRENT_DATE, MAX(last_purchase_date)) AS days_since_purchase,
        COUNT(DISTINCT DATE(created_at)) AS active_days
    FROM main.raw.customer_events
    WHERE created_date > CURRENT_DATE - INTERVAL 90 DAYS
    GROUP BY customer_id
""")

# Crear feature table
fs.create_table(
    name="main.features.customer_behavioral",
    primary_keys=["customer_id"],
    df=customer_features,
    description="Behavioral features for customer churn prediction (90-day window)",
    tags={"team": "data_science", "domain": "customer_analytics"}
)

print("✅ Feature table creada: main.features.customer_behavioral")
```

---

### 🎯 Usar Features en Training

```python
from databricks.feature_store import FeatureLookup

# Dataset con labels
labels_df = spark.table("main.labels.churn_labels")

# Lookup de features
feature_lookups = [
    FeatureLookup(
        table_name="main.features.customer_behavioral",
        feature_names=[
            "avg_session_time", 
            "purchase_count", 
            "total_spend",
            "days_since_purchase",
            "active_days"
        ],
        lookup_key="customer_id"
    )
]

# Crear training set (automáticamente hace join)
training_set = fs.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label="churn",
    exclude_columns=["created_date", "updated_date"]
)

# Convertir a pandas para sklearn
training_df = training_set.load_df().toPandas()

X = training_df.drop(["customer_id", "churn"], axis=1)
y = training_df["churn"]

# Entrenar
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

print("✅ Modelo entrenado con features del Feature Store")
```

---

### 🚀 Log Model con Feature Store

```python
# Log modelo + metadata de features
fs.log_model(
    model=model,
    artifact_path="churn_model",
    flavor=mlflow.sklearn,
    training_set=training_set,
    registered_model_name="churn_prediction_fs"
)

print("✅ Modelo registrado con lineage a Feature Store")
```

**¿Qué se guarda?**
* Modelo serializado
* **Feature metadata** (qué features usó)
* **Feature lineage** (de dónde vienen)
* **Lookup logic** (cómo hacer join)

---

### 📊 Scoring con Feature Store

#### Online Serving

```python
# Cargar modelo
model_uri = "models:/churn_prediction_fs/Production"
model = mlflow.pyfunc.load_model(model_uri)

# Scoring automáticamente hace lookup
new_customers = pd.DataFrame({
    'customer_id': [12345, 67890]
})

# Feature Store busca features automáticamente
predictions = fs.score_batch(
    model_uri=model_uri,
    df=spark.createDataFrame(new_customers)
)

display(predictions)
# customer_id | churn_probability
# 12345       | 0.15
# 67890       | 0.72
```

**¡No necesitas replicar lógica de features!**

---

### 🔄 Actualizar Features

```python
# Recalcular features con datos frescos
fresh_features = compute_features(new_data)

# Actualizar feature table (merge automático)
fs.write_table(
    name="main.features.customer_behavioral",
    df=fresh_features,
    mode="merge"  # Upsert basado en primary_key
)

print("✅ Features actualizadas")
```

**Modelos en producción** automáticamente usan features actualizadas.

---

### ✅ Ventajas del Feature Store

1. **Reutilización**: Una feature, múltiples modelos
2. **Consistencia**: Train = Serving (sin training-serving skew)
3. **Trazabilidad**: Lineage completo
4. **Colaboración**: Compartir features entre equipos
5. **Freshness**: Features online/offline
6. **Versionado**: Track cambios en features
7. **Discovery**: Buscar features existentes

---

### 📚 Casos de Uso

* 🛍️ **E-commerce**: `user_avg_cart_value`, `purchase_frequency`
* 🏛️ **Finanzas**: `account_balance_30d_avg`, `transaction_velocity`
* 📱 **SaaS**: `feature_usage_count`, `last_login_days_ago`
* 📦 **Logistica**: `delivery_time_avg`, `return_rate`

**Cualquier feature que se compute más de una vez → Feature Store**

## 📝 Conclusiones - MLflow y MLOps Completo

### 🎯 Key Takeaways

#### 1️⃣ **MLflow Tracking**

✅ **Registra TODO**:
* Parámetros (hiperparámetros)
* Métricas (accuracy, F1, RMSE)
* Modelos (serializados)
* Artefactos (plots, datasets)
* Código y entorno

💡 **Beneficio**: Comparar 100 experimentos en MLflow UI

---

#### 2️⃣ **Model Registry**

✅ **Gestiona lifecycle**:
```
None → Staging → Production → Archived
```

💡 **Beneficio**: Versionado + Rollback instantáneo

---

#### 3️⃣ **Deployment**

✅ **3 Patrones**:
* **Batch**: Scoring masivo periódico
* **Real-time API**: REST endpoint (<100ms)
* **Streaming**: Predicciones continuas

💡 **Beneficio**: Mismo modelo, múltiples formas de servir

---

#### 4️⃣ **Monitoreo**

✅ **Detectar degradación**:
* Data drift (KS test, Chi-squared)
* Concept drift (métricas caías)
* Schema drift (validación)

💡 **Beneficio**: Retraining automático cuando sea necesario

---

#### 5️⃣ **Feature Store**

✅ **Centraliza features**:
* Una feature, múltiples modelos
* Consistencia train/serving
* Trazabilidad completa

💡 **Beneficio**: No duplicar código, reutilizar trabajo

---

### 🚀 El Stack MLOps Completo

```
📊 DATA
   │
   └──→ 💾 FEATURE STORE (features centralizadas)
          │
          └──→ 📋 MLFLOW TRACKING (experimentos)
                 │
                 └──→ ⚖️  COMPARE (mejor modelo)
                        │
                        └──→ 📦 MODEL REGISTRY (versionado)
                               │
                               └──→ 🚀 DEPLOYMENT (batch/API/stream)
                                      │
                                      └──→ 📊 MONITORING (drift)
                                             │
                                             └──→ 🔄 RETRAIN (loop)
```

---

### 📈 Impacto en la Organización

| Métrica | Sin MLOps | Con MLOps |
|---------|-----------|------------|
| **Time to Production** | 6-12 meses | 2-4 semanas |
| **Modelos en Prod** | 1-3 | 10-50+ |
| **Experimentos/Semana** | 5-10 | 50-100 |
| **Rollback Time** | Días | Minutos |
| **Training-Serving Skew** | Común | Eliminado |
| **Colaboración** | Silos | Cross-funcional |

**ROI promedio de MLOps: 300-500%**

---

### 📚 Próximos Pasos

#### Nivel 1: Fundamentos ✅ (Completado)
* MLflow Tracking
* Model Registry
* Deployment básico

#### Nivel 2: Intermedio 🛣️ (Siguiente)
* CI/CD para ML (GitHub Actions + MLflow)
* Multi-model A/B testing
* Custom metrics y alerting
* Data quality monitoring

#### Nivel 3: Avanzado 🚀 (Futuro)
* Feature engineering at scale
* Real-time feature serving
* Model explainability (SHAP)
* Fairness y bias detection
* Multi-cloud deployment

---

### 📚 Recursos Recomendados

#### Documentación
* [MLflow Docs](https://mlflow.org/docs/latest/index.html)
* [Databricks Feature Store](https://docs.databricks.com/machine-learning/feature-store/)
* [Model Serving](https://docs.databricks.com/machine-learning/model-serving/)

#### Cursos
* **MLOps Specialization** (Coursera - DeepLearning.AI)
* **Databricks Academy**: Machine Learning Associate
* **Full Stack Deep Learning** (UC Berkeley)

#### Libros
* **"Introducing MLOps"** - Mark Treveil et al.
* **"Designing Machine Learning Systems"** - Chip Huyen
* **"Machine Learning Design Patterns"** - Lakshmanan et al.

---

### 🎉 ¡Felicidades!

**Has completado la sección de AutoML y MLOps.**

Ahora sabes:

✅ Entrenar modelos con Databricks AutoML  
✅ Trackear experimentos con MLflow  
✅ Gestionar modelos en producción  
✅ Detectar drift y reentrenar automáticamente  
✅ Usar Feature Store para consistencia  
✅ Deployar en batch, real-time, y streaming  

---

### 💡 Reflexión Final

> **"El 87% de los proyectos de ML nunca llegan a producción. MLOps cierra ese gap."**

> **"No necesitas el modelo perfecto. Necesitas un modelo BUENO en producción con monitoreo y mejora continua."**

**Fórmula del Éxito:**

$$\text{Impacto} = \text{Modelo} \times \text{Deployment} \times \text{Monitoring}$$

Sin deployment o monitoring, **Impacto = 0**.

---

### 🚀 Listo para Producción

**Tienes las herramientas. Ahora ¡a construir!** 💪

**Siguiente paso**: Aplicar esto a tu proyecto real. Empieza pequeño, itera rápido, escala cuando funcione.

🌟 **¡Éxito en tu journey de MLOps!** 🌟